# Feature Engineering Training v5 — seleksi RBF SVM tujuh fitur


## 1. Konfigurasi, lokasi, dan fungsi penyimpanan atomik


In [ ]:
from pathlib import Path
import hashlib, json, os, time
import joblib
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV, PredefinedSplit, train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.svm import SVC

SEED = 30092026
RUN_ID = "cifar10_7fitur_v5"
ROOT = Path.cwd().resolve()
OUT = ROOT / "outputs" / RUN_ID / "fusion"
MANIFEST_PATH = OUT / "laporan" / "combination_manifest.json"
LEDGER_PATH = OUT / "laporan" / "search_ledger.json"


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_object(payload):
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode()
    ).hexdigest()


def atomic_json(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    os.replace(temporary, path)


def atomic_joblib(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    joblib.dump(payload, temporary)
    os.replace(temporary, path)


if not MANIFEST_PATH.is_file():
    raise FileNotFoundError("Jalankan Feature Engineering Combination v5 dahulu")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
if manifest["run_id"] != RUN_ID or manifest["seed"] != SEED:
    raise ValueError("Manifest run/seed tidak sesuai")
print("Fusion v5:", OUT)


## 2. Validasi kedua arsip kombinasi


In [ ]:
features = {}
archive_paths = {}
for rep in ("A", "B"):
    path = OUT / "fitur" / f"train_combined_{rep}.npz"
    if (
        not path.is_file()
        or sha256_file(path) != manifest["combined_archive_hashes"][rep]
    ):
        raise ValueError("Arsip fusion hilang/berubah: " + rep)
    with np.load(path, allow_pickle=False) as data:
        if (
            str(data["run_id"]) != RUN_ID
            or str(data["representation"]) != rep
            or str(data["split_hash"]) != manifest["split_hash"]
            or not np.array_equal(data["boundaries"], manifest["boundaries"])
            or not np.array_equal(
                data["model_hashes"],
                [manifest["model_hashes"][v] for v in ("rgb", "avg", "ntsc")],
            )
        ):
            raise ValueError("Metadata kombinasi tidak cocok: " + rep)
        X = data["features"].astype(np.float32)
        labels = data["y"].astype(np.int64)
        indices = data["indices"].astype(np.int64)
        train_indices = data["train_idx"].astype(np.int64)
        val_indices = data["val_idx"].astype(np.int64)
    if X.shape != (50000, 2173) or not np.isfinite(X).all():
        raise ValueError("Bentuk/isi fitur tidak valid: " + rep)
    if rep == "A":
        y = labels
        idx_train = train_indices
        idx_val = val_indices
        if not np.array_equal(indices, np.arange(50000)):
            raise ValueError("Urutan sampel tidak cocok")
    elif (
        not np.array_equal(labels, y)
        or not np.array_equal(train_indices, idx_train)
        or not np.array_equal(val_indices, idx_val)
        or not np.array_equal(indices, np.arange(50000))
    ):
        raise ValueError("Representasi A/B tidak aligned")
    features[rep] = X
    archive_paths[rep] = path
if (
    hashlib.sha256(idx_train.tobytes() + idx_val.tobytes()).hexdigest()
    != manifest["split_hash"]
):
    raise ValueError("Split hash salah")
print("Dua representasi:", {k: v.shape for k, v in features.items()})


## 3. Definisi preprocessing per kelompok dan pipeline SVM


In [ ]:
class GroupPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, boundaries, weights, block_l2=False):
        self.boundaries = boundaries
        self.weights = weights
        self.block_l2 = block_l2

    def fit(self, X, y=None):
        self.boundaries_ = tuple(int(x) for x in self.boundaries)
        if self.boundaries_[0] != 0 or self.boundaries_[-1] != X.shape[1]:
            raise ValueError("Batas kelompok fitur salah")
        if len(self.weights) != len(self.boundaries_) - 1:
            raise ValueError("Jumlah bobot kelompok salah")
        self.scalers_ = [
            StandardScaler().fit(X[:, a:b])
            for a, b in zip(self.boundaries_[:-1], self.boundaries_[1:])
        ]
        return self

    def transform(self, X):
        if X.shape[1] != self.boundaries_[-1]:
            raise ValueError("Dimensi input salah")
        blocks = []
        for scaler, weight, a, b in zip(
            self.scalers_, self.weights, self.boundaries_[:-1], self.boundaries_[1:]
        ):
            block = scaler.transform(X[:, a:b]).astype(np.float32)
            if self.block_l2:
                block = Normalizer().transform(block)
            blocks.append(block * np.float32(weight))
        return Normalizer().transform(np.concatenate(blocks, axis=1)).astype(np.float32)


def matrix_for(spec):
    X = features[spec["rep"]]
    return X[:, :1536] if spec["groups"] == "cnn3" else X


def pipeline_for(spec):
    return Pipeline(
        [
            (
                "groups",
                GroupPreprocessor(
                    tuple(spec["boundaries"]), tuple(spec["weights"]), spec["block_l2"]
                ),
            ),
            (
                "svc",
                SVC(kernel="rbf", C=spec["C"], gamma=spec["gamma"], cache_size=1024),
            ),
        ]
    )


## 4. Screening subset dan identitas eksperimen


In [ ]:
screen_train, _ = train_test_split(
    idx_train, train_size=12000, stratify=y[idx_train], random_state=SEED
)
screen_val, _ = train_test_split(
    idx_val, train_size=3000, stratify=y[idx_val], random_state=SEED
)
screen_train, screen_val = np.sort(screen_train), np.sort(screen_val)
identity = {
    "run_id": RUN_ID,
    "seed": SEED,
    "manifest_sha256": sha256_file(MANIFEST_PATH),
    "archive_hashes": manifest["combined_archive_hashes"],
    "screen_train_sha256": hashlib.sha256(screen_train.tobytes()).hexdigest(),
    "screen_val_sha256": hashlib.sha256(screen_val.tobytes()).hexdigest(),
    "design": "v5-seven-group-rbf-median-gamma",
}
identity_sha = sha256_object(identity)
if LEDGER_PATH.exists():
    ledger = json.loads(LEDGER_PATH.read_text(encoding="utf-8"))
    if ledger["identity_sha256"] != identity_sha:
        raise ValueError("Ledger berasal dari fitur/konfigurasi lain")
else:
    ledger = {
        "identity_sha256": identity_sha,
        "identity": identity,
        "distance_diagnostics": {},
        "screen": {},
        "full": {},
    }
    atomic_json(LEDGER_PATH, ledger)
print("Identity:", identity_sha[:16], "| hasil screen:", len(ledger["screen"]))


## 5. Diagnostik jarak dan gamma dari training saja


In [ ]:
profiles = []
for cnn_weights in ((1.0, 1.0, 1.0), (2.0, 1.0, 1.0), (1.0, 0.7, 0.7)):
    for desc_weight in (0.1, 0.3):
        for block_l2 in (False, True):
            profiles.append(
                {
                    "rep": "B",
                    "groups": "all7",
                    "boundaries": (0, 512, 1024, 1536, 1860, 2116, 2164, 2173),
                    "weights": cnn_weights + (desc_weight,) * 4,
                    "block_l2": block_l2,
                }
            )
profiles.append(
    {
        "rep": "A",
        "groups": "all7",
        "boundaries": (0, 512, 1024, 1536, 1860, 2116, 2164, 2173),
        "weights": (1.0, 1.0, 1.0, 0.1, 0.1, 0.1, 0.1),
        "block_l2": False,
    }
)
profiles.append(
    {
        "rep": "B",
        "groups": "cnn3",
        "boundaries": (0, 512, 1024, 1536),
        "weights": (1.0, 1.0, 1.0),
        "block_l2": False,
    }
)
for profile in profiles:
    key = sha256_object(profile)[:20]
    if key in ledger["distance_diagnostics"]:
        continue
    X = matrix_for(profile)
    prep = GroupPreprocessor(
        tuple(profile["boundaries"]), tuple(profile["weights"]), profile["block_l2"]
    )
    Z = prep.fit_transform(X[screen_train])
    rng = np.random.default_rng(SEED)
    left = rng.integers(0, len(Z), size=3000)
    right = rng.integers(0, len(Z), size=3000)
    d2 = np.sum((Z[left] - Z[right]) ** 2, axis=1)
    d2 = d2[d2 > 1e-8]
    if len(d2) == 0:
        raise ValueError("Semua jarak fitur nol")
    median = float(np.median(d2))
    gamma_base = float(np.log(2.0) / median)
    ledger["distance_diagnostics"][key] = {
        "profile": profile,
        "squared_distance_p10": float(np.percentile(d2, 10)),
        "squared_distance_median": median,
        "squared_distance_p90": float(np.percentile(d2, 90)),
        "gamma_base": gamma_base,
    }
    atomic_json(LEDGER_PATH, ledger)
    print(
        "Jarak:",
        key,
        "median",
        round(median, 4),
        "gamma_base",
        round(gamma_base, 4),
        flush=True,
    )
print("Profil diagnostik:", len(ledger["distance_diagnostics"]))


## 6. GridSearchCV screening dengan checkpoint per kandidat


In [ ]:
search_idx = np.r_[screen_train, screen_val]
fold = PredefinedSplit(
    np.r_[np.full(12000, -1, dtype=np.int8), np.zeros(3000, dtype=np.int8)]
)
for profile in profiles:
    profile_key = sha256_object(profile)[:20]
    base = ledger["distance_diagnostics"][profile_key]["gamma_base"]
    gammas = ("scale", float(0.5 * base), float(base), float(2 * base))
    for C in (1.0, 10.0, 30.0):
        for gamma in gammas:
            spec = {**profile, "C": float(C), "gamma": gamma}
            key = sha256_object(spec)[:20]
            if key in ledger["screen"]:
                print("Screen cache:", key, flush=True)
                continue
            X = matrix_for(spec)
            grid = GridSearchCV(
                pipeline_for(spec),
                {"svc__C": [spec["C"]], "svc__gamma": [spec["gamma"]]},
                cv=fold,
                scoring="accuracy",
                refit=False,
                n_jobs=1,
                verbose=1,
                error_score="raise",
            )
            began = time.perf_counter()
            grid.fit(X[search_idx], y[search_idx])
            ledger["screen"][key] = {
                "spec": spec,
                "accuracy": float(grid.best_score_),
                "seconds": time.perf_counter() - began,
                "n_train": 12000,
                "n_val": 3000,
            }
            atomic_json(LEDGER_PATH, ledger)
            print("Screen:", key, "accuracy", f"{grid.best_score_:.2%}", flush=True)
print("Kandidat screen selesai:", len(ledger["screen"]))


## 7. Evaluasi empat kandidat tujuh fitur dan pembanding tiga CNN


In [ ]:
all7 = sorted(
    (item for item in ledger["screen"].values() if item["spec"]["groups"] == "all7"),
    key=lambda item: item["accuracy"],
    reverse=True,
)
cnn3 = sorted(
    (item for item in ledger["screen"].values() if item["spec"]["groups"] == "cnn3"),
    key=lambda item: item["accuracy"],
    reverse=True,
)
if len(all7) < 4 or not cnn3:
    raise RuntimeError("Screening belum cukup untuk evaluasi penuh")
full_specs = [item["spec"] for item in all7[:4]] + [cnn3[0]["spec"]]
for number, spec in enumerate(full_specs, 1):
    key = sha256_object(spec)[:20]
    if key in ledger["full"]:
        print("Full cache:", key, flush=True)
        continue
    X = matrix_for(spec)
    model = pipeline_for(spec)
    began = time.perf_counter()
    model.fit(X[idx_train], y[idx_train])
    pred = model.predict(X[idx_val])
    ledger["full"][key] = {
        "spec": spec,
        "accuracy": float(accuracy_score(y[idx_val], pred)),
        "macro_f1": float(f1_score(y[idx_val], pred, average="macro")),
        "correct": int(np.sum(y[idx_val] == pred)),
        "seconds": time.perf_counter() - began,
        "n_train": 40000,
        "n_val": 10000,
    }
    atomic_json(LEDGER_PATH, ledger)
    print(
        "Full",
        number,
        "/",
        len(full_specs),
        key,
        f"{ledger['full'][key]['accuracy']:.2%}",
        flush=True,
    )
print("Evaluasi penuh selesai:", len(ledger["full"]))


## 8. Bekukan konfigurasi berdasarkan validation


In [ ]:
all7_full = [
    item for item in ledger["full"].values() if item["spec"]["groups"] == "all7"
]
cnn3_full = [
    item for item in ledger["full"].values() if item["spec"]["groups"] == "cnn3"
]
if len(all7_full) < 4 or not cnn3_full:
    raise RuntimeError("Validasi penuh belum lengkap")
selected = max(all7_full, key=lambda item: item["accuracy"])
baseline = max(cnn3_full, key=lambda item: item["accuracy"])
selection = {
    "run_id": RUN_ID,
    "identity_sha256": identity_sha,
    "manifest_sha256": sha256_file(MANIFEST_PATH),
    "selected": selected,
    "cnn3_baseline": baseline,
    "target_validation_planning": 0.957,
    "post_exposure_testing": True,
}
selection = json.loads(json.dumps(selection))
selected = selection["selected"]
baseline = selection["cnn3_baseline"]
selection_path = OUT / "laporan" / "selection.json"
if selection_path.exists():
    old = json.loads(selection_path.read_text(encoding="utf-8"))
    if old != json.loads(json.dumps(selection)):
        raise ValueError("Seleksi final lama berbeda; jangan timpa")
else:
    atomic_json(selection_path, selection)
print("Fusion tujuh fitur:", f"{selected['accuracy']:.2%}")
print("Pembanding tiga CNN:", f"{baseline['accuracy']:.2%}")
print("Patokan 95,70% validation:", selected["accuracy"] >= 0.957)


## 9. Fit final preprocessing dan SVM pada 50K training


In [ ]:
model_path = OUT / "model" / "fusion_v5_final.joblib"
if model_path.exists():
    saved = joblib.load(model_path)
    if (
        saved["selection_sha256"] != sha256_file(selection_path)
        or saved["identity_sha256"] != identity_sha
        or saved["fit_count"] != 50000
    ):
        raise ValueError("Model final tidak cocok dengan seleksi")
    print("Model final yang cocok sudah ada:", model_path)
else:
    spec = selected["spec"]
    X = matrix_for(spec)
    final_pipeline = pipeline_for(spec)
    began = time.perf_counter()
    final_pipeline.fit(X, y)
    payload = {
        "run_id": RUN_ID,
        "pipeline": final_pipeline,
        "spec": spec,
        "fit_count": 50000,
        "selection_sha256": sha256_file(selection_path),
        "identity_sha256": identity_sha,
        "source_archive_sha256": manifest["combined_archive_hashes"][spec["rep"]],
        "fit_seconds": time.perf_counter() - began,
    }
    atomic_joblib(model_path, payload)
    print(
        "Fit final 50K selesai:", model_path, "detik:", round(payload["fit_seconds"], 1)
    )


## Akurasi training model fusion final 50K


In [ ]:
final_train_accuracy_path = OUT / "laporan" / "final_train_accuracy.json"
if final_train_accuracy_path.exists():
    final_train_accuracy = json.loads(
        final_train_accuracy_path.read_text(encoding="utf-8")
    )
    if final_train_accuracy.get("model_sha256") != sha256_file(
        model_path
    ) or final_train_accuracy.get("selection_sha256") != sha256_file(selection_path):
        raise ValueError("Akurasi training fusion tersimpan berasal dari model lain")
else:
    trained_model = joblib.load(model_path)
    if (
        trained_model.get("run_id") != RUN_ID
        or trained_model.get("fit_count") != 50000
        or trained_model.get("selection_sha256") != sha256_file(selection_path)
    ):
        raise ValueError("Model fusion final tidak cocok dengan seleksi")
    training_matrix = matrix_for(trained_model["spec"])
    train_predictions = trained_model["pipeline"].predict(training_matrix)
    final_train_accuracy = {
        "run_id": RUN_ID,
        "model_sha256": sha256_file(model_path),
        "selection_sha256": sha256_file(selection_path),
        "train_count": len(y),
        "train_correct": int(np.sum(train_predictions == y)),
        "train_accuracy": float(accuracy_score(y, train_predictions)),
    }
    atomic_json(final_train_accuracy_path, final_train_accuracy)
    del trained_model
print(
    "Fusion tujuh fitur + SVM final | training 50K:",
    f"{final_train_accuracy['train_accuracy']:.2%}",
)


## 10. Simpan ringkasan development


In [ ]:
summary = {
    "run_id": RUN_ID,
    "validation_all7": selected["accuracy"],
    "validation_cnn3": baseline["accuracy"],
    "validation_delta_all7_minus_cnn3": selected["accuracy"] - baseline["accuracy"],
    "target_validation_planning": 0.957,
    "selection_sha256": sha256_file(selection_path),
    "model_sha256": sha256_file(model_path),
    "number_screen_candidates": len(ledger["screen"]),
    "number_full_candidates": len(ledger["full"]),
    "post_exposure_testing": True,
}
report_path = OUT / "laporan" / "development_report.json"
if report_path.exists():
    old = json.loads(report_path.read_text(encoding="utf-8"))
    if old != summary:
        raise ValueError("Laporan development lama berbeda")
else:
    atomic_json(report_path, summary)
print("Laporan:", report_path)


## Tabel akurasi fusion pada tahap Training


In [ ]:
print(f"{'Model':<30} {'Training final 50K':>20} {'Validation 10K':>18}")
print("-" * 72)
print(
    f"{'Fusion tujuh fitur + SVM':<30} "
    f"{final_train_accuracy['train_accuracy']:>19.2%} "
    f"{selected['accuracy']:>17.2%}"
)
print(f"{'Pembanding tiga CNN':<30} {'-':>20} {baseline['accuracy']:>17.2%}")
print("Validation berasal dari model development 40K; model final dilatih pada 50K.")
